In [ ]:
# Practical 7: Dog Breed Classification using Transfer Learning
# Using MobileNetV2 pretrained on ImageNet

import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np

# ---------------------------------------------------------
# 1. Load Stanford Dogs Dataset
# ---------------------------------------------------------

(ds_train, ds_test), ds_info = tfds.load(
    'stanford_dogs',
    split=['train', 'test'],
    as_supervised=True,
    with_info=True
)

class_names = ds_info.features['label'].names

print("Total dog breeds:", len(class_names))
print("Using first 5 breeds:")
print(class_names[:5])

# ---------------------------------------------------------
# 2. Keep only 5 dog breeds for faster training
# ---------------------------------------------------------

NUM_CLASSES = 5

def filter_classes(image, label):
    return label < NUM_CLASSES

ds_train = ds_train.filter(filter_classes)
ds_test = ds_test.filter(filter_classes)

# ---------------------------------------------------------
# 3. Preprocess Images
# ---------------------------------------------------------

IMG_SIZE = 160
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    return image, label

ds_train = ds_train.map(preprocess)
ds_test = ds_test.map(preprocess)

# Split training data into training and validation
ds_train = ds_train.shuffle(1000, seed=42)

# FIX: Lowered from 4000 to 500 because 5 breeds only yield ~600 images total.
# This prevents val_data from being empty and crashing.
train_size = 500 

train_data = ds_train.take(train_size)
val_data = ds_train.skip(train_size)

train_data = train_data.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_data = val_data.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_data = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# ---------------------------------------------------------
# 4. Data Augmentation
# ---------------------------------------------------------

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1)
])

# ---------------------------------------------------------
# 5. Load Pretrained MobileNetV2
# ---------------------------------------------------------

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze pretrained layers
base_model.trainable = False

# ---------------------------------------------------------
# 6. Build Transfer Learning Model
# ---------------------------------------------------------

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = data_augmentation(inputs)

x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(x, training=False)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dense(128, activation='relu')(x)

x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

# ---------------------------------------------------------
# 7. Compile Model
# ---------------------------------------------------------

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ---------------------------------------------------------
# 8. Train Model
# ---------------------------------------------------------

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

# ---------------------------------------------------------
# 9. Fine-Tuning
# ---------------------------------------------------------

# Unfreeze the last 20 layers of MobileNetV2
base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nFine-tuning model...")

history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=3
)

# ---------------------------------------------------------
# 10. Evaluate Model
# ---------------------------------------------------------

test_loss, test_accuracy = model.evaluate(test_data)

print("\nTest Accuracy:", round(test_accuracy * 100, 2), "%")
print("Test Loss:", round(test_loss, 4))

# ---------------------------------------------------------
# 11. Display Prediction Results
# ---------------------------------------------------------

images, labels = next(iter(test_data))

predictions = model.predict(images, verbose=0)
predicted_labels = np.argmax(predictions, axis=1)

plt.figure(figsize=(12, 8))

for i in range(10):
    plt.subplot(2, 5, i + 1)

    # FIX: De-normalize image pixels back to [0, 255] for correct display.
    # Otherwise images will look corrupted/completely green-blue.
    img_to_show = (images[i].numpy() + 1) * 127.5
    plt.imshow(img_to_show.astype("uint8"))

    actual = class_names[labels[i].numpy()]
    predicted = class_names[predicted_labels[i]]

    plt.title(
        f"Actual: {actual}\nPredicted: {predicted}",
        fontsize=8
    )

    plt.axis("off")

plt.tight_layout()
plt.show()